# 12 — Step 10: the ranker

**Problem.** The best recommender so far (ALS + vibe blend, recall@10 0.0540) knows only taste and feel. It ignores whether a film is **good** (Step 4 quality) and whether people **know it today**. Adil's feedback, "I haven't seen most of these", points at the second gap. Its blend weight is also a single number tuned once, while PROJECT.md §4.3 says the tier-2 weight should be **learned by the ranker**.

**Two stages (how production recommenders work)**
1. **Candidates:** each user's top 300 from the ALS + vibe blend. This is cheap and broad, and its recall is the ceiling for stage 2, which can only reorder this list.
2. **Ranker:** reorder the 300 using all signals together, learned from data.

**Signals per (user, candidate)**

| Feature | Meaning |
|---|---|
| `als_z` | ALS taste score, standardised within the user's candidates |
| `vibe_z` | vibe similarity to the user's taste, standardised the same way |
| `blend_rank` | position in the current recommender (1 = top) |
| `shrunk`, `tier` | quality: Step 4 fair score and tier |
| `log_count` | all-time popularity (MovieLens ratings before the cutoff) |
| `log_recent` | **present-day popularity:** ratings in the ~18.5 months before the cutoff. The window length is the gap between the two cutoffs (2016-06-25 → 2018-01-04), not hand-picked |
| `year_gap` | distance between the film's year and the median year of films the user liked (PROJECT.md §7) |

**Rankers (roadmap), simplest first.** Each must beat the one before:
1. **blend:** the current best order (baseline);
2. **weighted score:** logistic regression on the features, a learned weighted sum;
3. **Random Forest** (bagging);
4. **LightGBM** (boosting, trained to rank: LambdaRank).

**No peeking**
- Rankers learn on the **2016 validation split**: 3,000 users, candidates from an ALS trained before mid-2016, labels = their 2016–2018 hits. Settings are tuned by cross-validation grouped by user.
- They are scored once on the **2018 testing zone** (5,000 users), against the blend.

**New metric: familiarity@10**, the mean present-day popularity percentile (0–1) of the top 10. It measures "would I recognise these?" alongside accuracy.

## 1. Load

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
import json
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..").resolve()))
from src.viberec import cf, evaluation as ev, ranker as rk, recommenders as rc

PROCESSED_DIR, ARTIFACTS_DIR, MODELS_DIR = Path("..") / "data" / "processed", Path("..") / "artifacts", Path("..") / "models"
SCOREBOARD = ARTIFACTS_DIR / "scoreboard.csv"
VERSION, SEED, K_SHRINK, N_CAND = "v1", 42, 7.324, 300


def write_metadata(path, **info):
    meta = {"created_at": datetime.now(timezone.utc).isoformat(), "produced_by": "notebooks/12_ranker.ipynb", **info}
    path.with_name(path.stem + ".metadata.json").write_text(json.dumps(meta, indent=2, default=str))


pre2018 = pd.read_parquet(PROCESSED_DIR / f"ratings_before_2018_{VERSION}.parquet")
early = pd.read_parquet(ARTIFACTS_DIR / f"q43_early_counts_{VERSION}.parquet")
VAL_CUT = int(early.loc[early["quantile"] == 0.8, "cutoff_ts"].iloc[0])
TEST_CUT = int(early.loc[early["quantile"] == 0.9, "cutoff_ts"].iloc[0])
WINDOW = TEST_CUT - VAL_CUT                                                  # ~18.5 months
movies = pd.read_parquet(PROCESSED_DIR / "movies_full.parquet")
movies["genres"] = movies["genres"].str.replace(",", "|", regex=False)
years = pd.read_parquet(PROCESSED_DIR / "movie_tiers_v2.parquet").set_index("movieId")["year"]
vibes = pd.read_parquet(PROCESSED_DIR / "vibe_scores_v2.parquet").set_index("movieId").drop(columns="source").astype(float)
tuning = pd.read_csv(ARTIFACTS_DIR / f"cf_tuning_{VERSION}.csv")
best_als = json.loads(tuning[tuning["method"] == "als"].sort_values(["recall@10", "recall@50"], ascending=False)["params"].iloc[0])
W_ALS = float(pd.read_csv(ARTIFACTS_DIR / f"cf_vibe_blend_tuning_{VERSION}.csv").sort_values(["recall@10", "recall@50"], ascending=False)["w_als"].iloc[0])
print(f"window {WINDOW / 86400 / 30.44:.1f} months | ALS {best_als} | blend w_als {W_ALS}")

splits = {}
for name, cut, q, train_p, hits_p in (("val", VAL_CUT, 0.8, "validation_train", "validation_hits"),
                                      ("test", TEST_CUT, 0.9, "benchmark_train", "benchmark_hits")):
    tr = pd.read_parquet(PROCESSED_DIR / f"{train_p}_{VERSION}.parquet")
    known = ev.movies_known_at(early[early["quantile"] == q], movies, K_SHRINK, pd.to_datetime(cut, unit="s").year)
    mt = rk.movie_table(known, rk.recent_counts(pre2018, cut, WINDOW), years)
    splits[name] = {"cut": cut, "train": tr, "mean": tr.groupby("userId")["rating"].mean(), "movies": mt,
                    "hits": pd.read_parquet(PROCESSED_DIR / f"{hits_p}_{VERSION}.parquet").groupby("userId")["movieId"].apply(set).to_dict(),
                    "pool": mt.index[mt["tier"] == 3].to_numpy()}
    print(f"{name}: {len(splits[name]['hits']):,} users, pool {len(splits[name]['pool']):,} tier-3 movies")

## 2. ALS for each cutoff, candidates and feature rows

- The 2018 ALS model is the one saved in notebook 09.
- The 2016 one is trained here on pre-2016 ratings with the same settings, and saved.
- For every user: their top 300 by the blend, excluding movies they rated, with one feature row per candidate and the label "became a hit". Rows are cached to `artifacts/`.

In [ ]:
from implicit.als import AlternatingLeastSquares


def als_for(name):
    ratings = pre2018 if name == "test" else pre2018[pre2018["timestamp"] < VAL_CUT]
    m, ui, mi = cf.build_matrix(ratings, best_als["table"])
    if name == "test":
        itf, uf = np.load(MODELS_DIR / f"als_item_factors_{VERSION}.npy"), np.load(MODELS_DIR / f"als_user_factors_{VERSION}.npy")
    else:
        p_i, p_u = MODELS_DIR / f"als_item_factors_pre2016_{VERSION}.npy", MODELS_DIR / f"als_user_factors_pre2016_{VERSION}.npy"
        if p_i.exists():
            itf, uf = np.load(p_i), np.load(p_u)
        else:
            start = time.time()
            model = AlternatingLeastSquares(factors=best_als["factors"], regularization=best_als["reg"], alpha=best_als["alpha"],
                                            iterations=15, random_state=SEED)
            model.fit(m, show_progress=False)
            itf, uf = np.asarray(model.item_factors), np.asarray(model.user_factors)
            np.save(p_i, itf); np.save(p_u, uf)
            write_metadata(p_i, params=best_als, trained_on="ratings before 2016-06-25", runtime_seconds=round(time.time() - start, 1))
    assert len(itf) == len(mi) and len(uf) == len(ui)
    return itf, uf, ui, mi


def build_rows(name):
    path = ARTIFACTS_DIR / f"ranker_rows_{name}_{VERSION}.parquet"
    if path.exists():
        print(f"{path} already exists, loading.")
        return pd.read_parquet(path)
    sp = splits[name]
    itf, uf, ui, mi = als_for(name)
    als = cf.factor_user_scores(uf, itf, ui)
    vib = cf.vibe_user_scores(vibes, mi, sp["mean"])
    blend = cf.blend_user_scores([als, vib], [W_ALS, 1 - W_ALS])
    pool = np.asarray([m for m in sp["pool"] if m in mi.index])
    pool_cols = mi.loc[pool].to_numpy()
    by_user = dict(tuple(sp["train"].groupby("userId")))
    parts, start = [], time.time()
    for user, hits_u in sp["hits"].items():
        r = by_user[user]
        a, v = als(user, r), vib(user, r)
        b = cf.blend_user_scores([lambda *_: a, lambda *_: v], [W_ALS, 1 - W_ALS])(user, r)
        s = b[pool_cols].copy()
        s[np.isin(pool, r["movieId"].to_numpy())] = -np.inf
        top = np.argpartition(-s, N_CAND)[:N_CAND]
        cand = pool[top[np.argsort(-s[top])]]
        rows = rk.user_rows(user, r, sp["mean"][user], cand, a, v, b, sp["movies"], mi)
        rows["label"] = rows["movieId"].isin(hits_u).astype(int)
        parts.append(rows)
    out = pd.concat(parts, ignore_index=True)
    out.to_parquet(path, index=False)
    write_metadata(path, split=name, users=len(sp["hits"]), candidates_per_user=N_CAND, features=rk.FEATURES,
                   runtime_seconds=round(time.time() - start, 1))
    return out


rows = {name: build_rows(name) for name in ("val", "test")}
for name, r in rows.items():
    hits_total = sum(len(h) for h in splits[name]["hits"].values())
    print(f"{name}: {len(r):,} rows; candidates contain {r['label'].sum():,} of {hits_total:,} future hits "
          f"(ceiling recall@300 {r.groupby('userId')['label'].sum().sum() / hits_total:.1%}); positives {r['label'].mean():.2%}")

## 3. Train the rankers on the 2016 validation users

Settings are chosen by 3-fold cross-validation **grouped by user** (a user's candidates are never split between training and scoring), by recall@10 with ties broken by recall@50, the same rule as before.

In [ ]:
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

val = rows["val"].fillna(0).sort_values(["userId", "blend_rank"]).reset_index(drop=True)
X, y, groups = val[rk.FEATURES].to_numpy(), val["label"].to_numpy(), val["userId"].to_numpy()
val_hits = splits["val"]["hits"]


def recall_on(frame, score, k):
    recs = rk.top_k_by(frame, score, k=max(k, 50))
    return np.mean([ev.recall_at_k(recs.get(u, []), val_hits[u], k) for u in frame["userId"].unique()])


def make_model(kind, params):
    if kind == "weighted":
        return make_pipeline(StandardScaler(), LogisticRegression(C=params["C"], max_iter=2000))
    if kind == "forest":
        return RandomForestClassifier(n_estimators=200, min_samples_leaf=params["leaf"], max_features="sqrt", n_jobs=-1, random_state=SEED)
    return lgb.LGBMRanker(objective="lambdarank", n_estimators=params["trees"], num_leaves=params["leaves"],
                          learning_rate=0.05, min_child_samples=50, random_state=SEED, verbose=-1)


def fit_predict(kind, params, tr_idx, te_idx):
    model = make_model(kind, params)
    if kind == "lightgbm":
        tr = val.iloc[tr_idx]
        model.fit(tr[rk.FEATURES].to_numpy(), tr["label"].to_numpy(), group=tr.groupby("userId", sort=False).size().to_numpy())
        return model, model.predict(val.iloc[te_idx][rk.FEATURES].to_numpy())
    model.fit(X[tr_idx], y[tr_idx])
    return model, model.predict_proba(X[te_idx])[:, 1]


GRIDS = {"weighted": [{"C": c} for c in (0.01, 0.1, 1.0)],
         "forest": [{"leaf": l} for l in (20, 100, 500)],
         "lightgbm": [{"leaves": nl, "trees": t} for nl in (15, 63) for t in (200, 600)]}
TUNE_PATH = ARTIFACTS_DIR / f"ranker_tuning_{VERSION}.csv"
done = pd.read_csv(TUNE_PATH) if TUNE_PATH.exists() else pd.DataFrame(columns=["kind", "params"])
folds = list(GroupKFold(3).split(X, y, groups))
for kind, grid in GRIDS.items():
    for params in grid:
        key = json.dumps(params, sort_keys=True)
        if ((done["kind"] == kind) & (done["params"] == key)).any():
            continue
        start, r10, r50 = time.time(), [], []
        for tr_idx, te_idx in folds:
            _, s = fit_predict(kind, params, tr_idx, te_idx)
            part = val.iloc[te_idx]
            r10.append(recall_on(part, s, 10)); r50.append(recall_on(part, s, 50))
        row = {"kind": kind, "params": key, "recall@10": np.mean(r10), "recall@50": np.mean(r50), "seconds": round(time.time() - start, 1)}
        done = pd.concat([done, pd.DataFrame([row])], ignore_index=True)
        done.to_csv(TUNE_PATH, index=False)
        print(f"{kind:9s} {key:30s} cv recall@10 {row['recall@10']:.4f} recall@50 {row['recall@50']:.4f} ({row['seconds']:.0f}s)")
blend_cv = np.mean([recall_on(val.iloc[te], -val.iloc[te]["blend_rank"].to_numpy(), 10) for _, te in folds])
print(f"blend (no ranker) on the same folds: recall@10 {blend_cv:.4f}")
tuned = pd.read_csv(TUNE_PATH).sort_values(["recall@10", "recall@50"], ascending=False).groupby("kind").head(1).set_index("kind")
print(tuned.round(4).to_string())

## 4. Testing zone: the tuned rankers on the 2018 users

Each ranker is refitted on all validation rows with its tuned settings, applied to the 2018 candidates, and scored against the blend. Results are added to the scoreboard.

In [ ]:
test = rows["test"].fillna(0).sort_values(["userId", "blend_rank"]).reset_index(drop=True)
test_hits, test_mean = splits["test"]["hits"], splits["test"]["mean"]
recent_pct = splits["test"]["movies"]["recent_pct"].to_dict()
pop_share = (splits["test"]["movies"]["rating_count"] / splits["test"]["movies"]["rating_count"].sum()).to_dict()
z = rc.standardise(vibes)
vibe_vectors = dict(zip(z.index, z.to_numpy()))
all_idx = np.arange(len(val))
scores = {"blend (no ranker)": -test["blend_rank"].to_numpy()}
models = {}
for kind in ("weighted", "forest", "lightgbm"):
    params = json.loads(tuned.loc[kind, "params"])
    model = make_model(kind, params)
    if kind == "lightgbm":
        model.fit(X, y, group=val.groupby("userId", sort=False).size().to_numpy())
        scores[kind] = model.predict(test[rk.FEATURES].to_numpy())
    else:
        model.fit(X, y)
        scores[kind] = model.predict_proba(test[rk.FEATURES].to_numpy())[:, 1]
    models[kind] = model

per_user, results = {}, {}
for name, s in scores.items():
    recs = rk.top_k_by(test, s, k=50)
    rows_u = []
    for u, h in test_hits.items():
        r = recs.get(u, [])
        rows_u.append({"userId": u, "recall@10": ev.recall_at_k(r, h, 10), "recall@50": ev.recall_at_k(r, h, 50),
                       "ndcg@10": ev.ndcg_at_k(r, h, 10), "familiarity@10": rk.familiarity(r, recent_pct),
                       "novelty@10": ev.novelty(r, pop_share, 10), "diversity@10": ev.diversity(r, vibe_vectors, 10)})
    per_user[name] = pd.DataFrame(rows_u).set_index("userId")
    results[name] = per_user[name].mean().to_dict()
    results[name]["coverage@10"] = ev.coverage(list(recs.values()), len(splits["test"]["pool"]), 10)
    ev.append_scoreboard(SCOREBOARD, "10 ranker", f"{name} over ALS+vibe top {N_CAND}", results[name],
                         settings={"cutoff": "2018-01-04", "users": len(test_hits), "seed": SEED, "candidates": N_CAND,
                                   "params": tuned.loc[name, "params"] if name in tuned.index else "none"},
                         note="ranker trained on the 2016 validation split")
print(pd.DataFrame(results).T[["recall@10", "recall@50", "ndcg@10", "familiarity@10", "coverage@10", "novelty@10", "diversity@10"]].round(4).to_string())
print()
for a, b in [("weighted", "blend (no ranker)"), ("forest", "weighted"), ("lightgbm", "weighted"), ("lightgbm", "blend (no ranker)")]:
    parts = []
    for metric in ("recall@10", "ndcg@10", "familiarity@10"):
        d, lo, hi, rel = ev.paired_gain(per_user[a], per_user[b], metric, seed=SEED)
        parts.append(f"{metric} {rel:+.0%} [{lo:+.4f}, {hi:+.4f}]")
    print(f"{a} vs {b}: " + " | ".join(parts))

## 5. What did the rankers learn?

- **Weighted score:** the coefficients on standardised features (sign and size = direction and strength).
- **LightGBM:** gain importance (how much each feature improved its decisions).

These are also the raw material for the Torch button (Step 11).

In [ ]:
coef = pd.Series(models["weighted"][-1].coef_[0], index=rk.FEATURES).sort_values()
print("weighted score coefficients (standardised features):\n", coef.round(3).to_string())
gain = pd.Series(models["lightgbm"].booster_.feature_importance("gain"), index=rk.FEATURES)
print("\nLightGBM gain share:\n", (gain / gain.sum()).sort_values(ascending=False).round(3).to_string())

**Results (2018 testing zone, 5,000 users; candidates = top 300 of the ALS + vibe blend, containing 30.9% of all future hits, the ceiling)**

| Ranker | recall@10 | recall@50 | ndcg@10 | coverage@10 | diversity@10 |
|---|---|---|---|---|---|
| blend (no ranker, previous best) | 0.0540 | 0.1615 | 0.1440 | 6.7% | 0.45 |
| weighted score (logistic regression) | 0.0620 (+15%) | 0.1812 | 0.1599 | 4.0% | 0.51 |
| Random Forest | 0.0726 | 0.1937 | 0.1786 | 5.6% | 0.47 |
| **LightGBM (LambdaRank)** | **0.0764 (+41%)** | **0.2026** | **0.1802** | 4.6% | 0.49 |

Paired comparisons (95% intervals):
- **weighted vs blend:** recall@10 +15% [+0.0057, +0.0101], ndcg@10 +11%;
- **Random Forest vs weighted:** +17%;
- **LightGBM vs weighted:** +23% [+0.0114, +0.0174];
- **LightGBM vs blend:** recall@10 +41% [+0.0193, +0.0254], ndcg@10 +25%.

Each technique beats the simpler one, so by the rule **LightGBM earns its place**. It is also about 3× faster to train than the Random Forest. Validation cross-validation agreed: LightGBM 0.093 > forest 0.088 > weighted 0.067 > blend 0.060.

**What the rankers learned (this answers Adil's "I haven't seen these")**
- **Present-day popularity is the strongest signal:** 39% of LightGBM's gain, and the largest positive weight in the weighted score (+0.81).
- **All-time popularity counts negatively once present-day popularity is known** (−0.43). Films people watch *now* are favoured; old classics that people stopped watching are not. This matches Adil's preference for recent, recognisable films, and the data found it on its own.
- **Then** the blend position, ALS taste, quality (fair score), year gap and vibe.
- **Tier carries no weight:** every candidate here is tier 3, so the ranker cannot learn a tier-2 weight yet. PROJECT.md §4.3 item 9 needs tier-2 candidates included; a next step.

**Costs and limitations**
- **Coverage drops** (6.7% → 4.6% of the catalog appears in someone's top 10): leaning on present-day popularity concentrates recommendations.
- **familiarity@10 is not informative here:** it is ≈ 0.99 for every method, because personalised candidates are already among the most-watched movies of the 30,000-movie pool. Adil's unfamiliar results came from **search**, which had no popularity signal; the ranker's lesson (present-day popularity matters most) is what the search needs next.
- **The ranker is capped by the candidates:** only 30.9% of future hits are in the top 300.